In [2]:
import pandas as pd

df1 = pd.read_csv('/kaggle/input/datasets/sid321axn/malicious-urls-dataset/malicious_phish.csv', nrows=5)
print("Dataset 1 columns:", df1.columns.tolist())

df2 = pd.read_csv('/kaggle/input/datasets/sslliiman/phishstorm-phishing-legitimate-url-dataset/correct phishing.csv', nrows=5)
print("Dataset 2 columns:", df2.columns.tolist())

Dataset 1 columns: ['url', 'type']
Dataset 2 columns: ['url', 'type', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14']


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import traceback

# ============================================================================
# CONFIGURATION - UPDATE THESE PATHS (KAGGLE COMPATIBLE)
# ============================================================================

DATASET_1_PATH = '/kaggle/input/datasets/sid321axn/malicious-urls-dataset/malicious_phish.csv'
DATASET_2_PATH = '/kaggle/input/datasets/sslliiman/phishstorm-phishing-legitimate-url-dataset/correct phishing.csv'
OUTPUT_PATH = '/kaggle/working/corrected_phishing_dataset.csv'

# Column names - based on your datasets
URL_COLUMN_DATASET_1 = 'url'
LABEL_COLUMN_DATASET_1 = 'type'

URL_COLUMN_DATASET_2 = 'url'
LABEL_COLUMN_DATASET_2 = 'type'

# ============================================================================
# ROBUST FILE LOADING WITH ENCODING FALLBACK (NO EXTERNAL DEPS)
# ============================================================================

def load_csv_with_fallback_encoding(file_path, encoding_attempts=None):
    """
    Try multiple encodings to load CSV file
    Falls back through common encodings if primary fails
    """
    if encoding_attempts is None:
        encoding_attempts = [
            'utf-8',
            'latin-1',      # Also called iso-8859-1
            'iso-8859-1',
            'cp1252',       # Windows encoding (common for non-English files)
            'utf-8-sig',    # UTF-8 with BOM
            'utf-16',
        ]
    
    print(f"\nAttempting to load: {Path(file_path).name}")
    print(f"File path: {file_path}")
    
    # Check if file exists
    if not Path(file_path).exists():
        print(f"✗ ERROR: File not found!")
        return None, None
    
    file_size = Path(file_path).stat().st_size
    print(f"File size: {file_size:,} bytes")
    
    for i, encoding in enumerate(encoding_attempts, 1):
        try:
            print(f"  [{i}/{len(encoding_attempts)}] Trying {encoding:<12}...", end=" ", flush=True)
            df = pd.read_csv(
                file_path, 
                encoding=encoding, 
                on_bad_lines='skip',
                engine='python'  # More robust parser for problematic files
            )
            print(f"✓ SUCCESS!")
            print(f"       → Loaded: {len(df):,} rows, {len(df.columns)} columns")
            return df, encoding
        except (UnicodeDecodeError, UnicodeError):
            print(f"✗ (encoding)")
        except Exception as e:
            print(f"✗ ({type(e).__name__})")
    
    print(f"\n✗ ERROR: Could not load file with any tested encoding!")
    print(f"  Encodings tried: {', '.join(encoding_attempts)}")
    return None, None

def load_datasets_robust(path_1, path_2):
    """Load both datasets with robust encoding handling"""
    
    print("="*75)
    print("LOADING DATASETS WITH ENCODING RECOVERY")
    print("="*75)
    
    # Load Dataset 1
    print(f"\n[Dataset 1 - Main Dataset (651K URLs)]")
    df_1, enc_1 = load_csv_with_fallback_encoding(path_1)
    if df_1 is None:
        print("✗ Failed to load Dataset 1")
        return None, None
    
    print(f"  ✓ Encoding used: {enc_1}")
    print(f"  ✓ Columns: {df_1.columns.tolist()}")
    
    # Load Dataset 2
    print(f"\n[Dataset 2 - Reference Dataset (100K URLs)]")
    df_2, enc_2 = load_csv_with_fallback_encoding(path_2)
    if df_2 is None:
        print("✗ Failed to load Dataset 2")
        return None, None
    
    print(f"  ✓ Encoding used: {enc_2}")
    print(f"  ✓ Columns: {df_2.columns.tolist()}")
    
    return df_1, df_2

# ============================================================================
# DATA VALIDATION
# ============================================================================

def validate_columns(df, dataset_name, url_col, label_col):
    """Validate that required columns exist"""
    if url_col not in df.columns:
        print(f"\n✗ ERROR: Column '{url_col}' not found in {dataset_name}")
        print(f"  Available columns: {df.columns.tolist()}")
        return False
    
    if label_col not in df.columns:
        print(f"\n✗ ERROR: Column '{label_col}' not found in {dataset_name}")
        print(f"  Available columns: {df.columns.tolist()}")
        return False
    
    return True

# ============================================================================
# CORRECTION LOGIC
# ============================================================================

def normalize_urls(url):
    """Normalize URLs for comparison (lowercase, strip whitespace)"""
    return str(url).strip().lower()

def correct_dataset_1(df_1, df_2, url_col_1, label_col_1, 
                      url_col_2, label_col_2):
    """
    Correct Dataset 1 using Dataset 2 as reference
    
    Logic:
    - Create a mapping of URLs from Dataset 2 with their correct labels
    - Find matching URLs in Dataset 1
    - If a URL exists in Dataset 2, use Dataset 2's label as ground truth
    - Focus on correcting misclassified phishing URLs
    """
    
    print("\n" + "="*75)
    print("STARTING CORRECTION PROCESS")
    print("="*75)
    
    # Normalize URLs for comparison
    df_1_copy = df_1.copy()
    df_2_copy = df_2.copy()
    
    print("\nNormalizing URLs in Dataset 1...")
    df_1_copy['url_normalized'] = df_1_copy[url_col_1].apply(normalize_urls)
    
    print("Normalizing URLs in Dataset 2...")
    df_2_copy['url_normalized'] = df_2_copy[url_col_2].apply(normalize_urls)
    
    # Remove duplicates in Dataset 2 (keep first occurrence)
    print(f"Removing duplicates in Dataset 2...")
    before_dedup = len(df_2_copy)
    df_2_copy = df_2_copy.drop_duplicates(subset=['url_normalized'], keep='first')
    duplicates_removed = before_dedup - len(df_2_copy)
    print(f"  → Duplicates removed: {duplicates_removed:,}")
    print(f"  → Reference URLs: {len(df_2_copy):,}")
    
    # Create reference mapping from Dataset 2
    reference_dict = dict(zip(df_2_copy['url_normalized'], 
                              df_2_copy[label_col_2]))
    
    # Track corrections
    corrections_made = {
        'phishing_to_benign': 0,
        'phishing_to_other': 0,
        'other_corrected': 0,
        'total_matched': 0
    }
    
    # Apply corrections
    print("\nApplying corrections...")
    for idx, row in df_1_copy.iterrows():
        url_norm = row['url_normalized']
        current_label = row[label_col_1]
        
        if url_norm in reference_dict:
            corrections_made['total_matched'] += 1
            correct_label = reference_dict[url_norm]
            
            # If current label differs from reference, correct it
            if current_label != correct_label:
                if current_label == 'phishing' and correct_label == 'benign':
                    corrections_made['phishing_to_benign'] += 1
                elif current_label == 'phishing':
                    corrections_made['phishing_to_other'] += 1
                else:
                    corrections_made['other_corrected'] += 1
                
                df_1_copy.loc[idx, label_col_1] = correct_label
        
        # Progress indicator every 100K rows
        if (idx + 1) % 100000 == 0:
            print(f"  ✓ Processed {idx + 1:,} / {len(df_1_copy):,} URLs")
    
    # Remove the normalized column
    df_1_copy = df_1_copy.drop(columns=['url_normalized'])
    
    return df_1_copy, corrections_made

# ============================================================================
# ANALYSIS AND REPORTING
# ============================================================================

def compare_datasets(df_original, df_corrected, label_col):
    """Compare original and corrected datasets"""
    print("\n" + "="*75)
    print("DATASET COMPARISON - BEFORE & AFTER")
    print("="*75)
    
    print("\n[Before Correction - Dataset 1]")
    print(df_original[label_col].value_counts().to_string())
    
    print("\n[After Correction - Dataset 1]")
    print(df_corrected[label_col].value_counts().to_string())

def generate_correction_report(df_1_original, df_1_corrected, corrections_made, 
                              label_col):
    """Generate detailed correction report"""
    print("\n" + "="*75)
    print("CORRECTION REPORT - DETAILED STATISTICS")
    print("="*75)
    
    print(f"\n[Dataset Size]")
    print(f"  Total URLs in Dataset 1: {len(df_1_original):,}")
    
    print(f"\n[Matching & Corrections]")
    print(f"  URLs matched with Dataset 2: {corrections_made['total_matched']:,} ({corrections_made['total_matched']/len(df_1_original)*100:.2f}%)")
    print(f"  Total corrections made: {sum(corrections_made.values()) - corrections_made['total_matched']:,}")
    
    print(f"\n[Correction Breakdown]")
    print(f"  • Phishing → Benign: {corrections_made['phishing_to_benign']:,}")
    print(f"  • Phishing → Other types: {corrections_made['phishing_to_other']:,}")
    print(f"  • Other corrections: {corrections_made['other_corrected']:,}")
    
    total_corrections = (corrections_made['phishing_to_benign'] + 
                        corrections_made['phishing_to_other'] + 
                        corrections_made['other_corrected'])
    print(f"\n  TOTAL CORRECTIONS: {total_corrections:,} ({total_corrections/len(df_1_original)*100:.2f}%)")
    
    print(f"\n[Label Distribution Change]")
    print(f"  Before correction:")
    for label, count in df_1_original[label_col].value_counts().items():
        print(f"    - {label}: {count:,}")
    
    print(f"\n  After correction:")
    for label, count in df_1_corrected[label_col].value_counts().items():
        print(f"    - {label}: {count:,}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function"""
    
    try:
        # Load datasets with robust encoding
        df_1, df_2 = load_datasets_robust(DATASET_1_PATH, DATASET_2_PATH)
        
        if df_1 is None or df_2 is None:
            print("\n✗ Failed to load datasets!")
            return False
        
        # Validate columns exist
        print("\n" + "="*75)
        print("VALIDATING COLUMN NAMES")
        print("="*75)
        
        if not validate_columns(df_1, "Dataset 1", URL_COLUMN_DATASET_1, LABEL_COLUMN_DATASET_1):
            return False
        if not validate_columns(df_2, "Dataset 2", URL_COLUMN_DATASET_2, LABEL_COLUMN_DATASET_2):
            return False
        
        print("\n✓ All columns validated successfully!")
        
        # Perform correction
        df_1_corrected, corrections = correct_dataset_1(
            df_1, df_2,
            URL_COLUMN_DATASET_1, LABEL_COLUMN_DATASET_1,
            URL_COLUMN_DATASET_2, LABEL_COLUMN_DATASET_2
        )
        
        # Compare and report
        compare_datasets(df_1, df_1_corrected, LABEL_COLUMN_DATASET_1)
        generate_correction_report(df_1, df_1_corrected, corrections, LABEL_COLUMN_DATASET_1)
        
        # Save corrected dataset
        print("\n" + "="*75)
        print("SAVING CORRECTED DATASET")
        print("="*75)
        print(f"\nSaving to: {OUTPUT_PATH}")
        df_1_corrected.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
        print(f"✓ Corrected dataset saved successfully!")
        print(f"  File size: {Path(OUTPUT_PATH).stat().st_size:,} bytes")
        
        return True
        
    except Exception as e:
        print(f"\n✗ ERROR: {e}")
        traceback.print_exc()
        return False

# ============================================================================
# RUN THE SCRIPT
# ============================================================================

if __name__ == "__main__":
    success = main()

    if success:
        print("\n" + "="*75)
        print("✓ CORRECTION COMPLETED SUCCESSFULLY!")
        print("="*75)
    else:
        print("\n" + "="*75)
        print("✗ CORRECTION FAILED - CHECK ERRORS ABOVE")
        print("="*75)

LOADING DATASETS WITH ENCODING RECOVERY

[Dataset 1 - Main Dataset (651K URLs)]

Attempting to load: malicious_phish.csv
File path: /kaggle/input/datasets/sid321axn/malicious-urls-dataset/malicious_phish.csv
File size: 45,664,439 bytes
  [1/6] Trying utf-8       ... ✓ SUCCESS!
       → Loaded: 651,191 rows, 2 columns
  ✓ Encoding used: utf-8
  ✓ Columns: ['url', 'type']

[Dataset 2 - Reference Dataset (100K URLs)]

Attempting to load: correct phishing.csv
File path: /kaggle/input/datasets/sslliiman/phishstorm-phishing-legitimate-url-dataset/correct phishing.csv
File size: 8,401,696 bytes
  [1/6] Trying utf-8       ... ✗ (encoding)
  [2/6] Trying latin-1     ... ✓ SUCCESS!
       → Loaded: 96,011 rows, 15 columns
  ✓ Encoding used: latin-1
  ✓ Columns: ['url', 'type', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14']

VALIDATING COLUMN NAMES

✓ All col